## 2D Grid for XZZX Surface Code Format

In [1]:
import numpy as np
# create mapping
n = 21  # number of qubits

rows = 5
cols = 5
corners = [(0, 0), (0, cols - 1), (rows - 1, 0), (rows - 1, cols - 1)]
mapping = []
qubit_index = 0
for i in range(rows):
    for j in range(cols):
        if (i, j) not in corners:
            mapping.append([i, j])
            qubit_index +=1

mapping = np.array(mapping)
print(mapping)


[[0 1]
 [0 2]
 [0 3]
 [1 0]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [3 4]
 [4 1]
 [4 2]
 [4 3]]


In [2]:

# 2D naive, stabilizer to grids
import numpy as np

grid_X = np.zeros((rows, cols))
stabs = np.array([0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,  
                  0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1])
print(f"length of stabs: {len(stabs)}")
for i in range(0, n):
    row, col = mapping[i]
    grid_X[row, col] = stabs[i]


grid_Z = np.zeros((rows, cols))
for i in range(n, 2 * n):
    row, col = mapping[i-n]
    grid_Z[row, col] = stabs[i]

length of stabs: 42


In [3]:
grid_X

array([[0., 0., 1., 1., 0.],
       [1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1.],
       [1., 0., 1., 1., 1.],
       [0., 1., 1., 1., 0.]])

In [4]:
grid_Z

array([[0., 0., 1., 1., 0.],
       [1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 0.],
       [0., 0., 1., 1., 0.]])

## JIT-able numpy version

In [5]:
import numpy as np

# stabs.shape == (num_samples, 2 * n)
# mapping.shape == (n, 2), containing (row, col) for each qubit
rows = 5
cols = 5
mapping = np.asarray(mapping)
row_idx, col_idx = mapping.T
print(f"row indices: {row_idx}")
print(f"col indices: {col_idx}")

num_samples = stabs.shape[0]
tableau = np.stack([stabs] * n)
print(f"tableau shape: {tableau.shape}")


grid_X = np.zeros((n, rows, cols), dtype=stabs.dtype)  # n stabilizer rows
grid_Z = np.zeros((n, rows, cols), dtype=stabs.dtype)  # n stabilizer rows

print(f"grid_X shape: {grid_X.shape}, grid_Z shape: {grid_Z.shape}")

# # First n entries in every stabilizer row go to X
grid_X[:, row_idx, col_idx] = tableau[:, :n]

# # Next n entries in every stabilizer row go to Z
grid_Z[:, row_idx, col_idx] = tableau[:, n:2*n]

row indices: [0 0 0 1 1 1 1 1 2 2 2 2 2 3 3 3 3 3 4 4 4]
col indices: [1 2 3 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 1 2 3]
tableau shape: (21, 42)
grid_X shape: (21, 5, 5), grid_Z shape: (21, 5, 5)


In [6]:
grid_X

array([[[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0, 1, 1, 1, 0]],

       [[0, 0, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1],
        [1, 0, 1, 1, 1],
        [0,

## Actually JIT-able version

In [7]:
! pip install jax



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import jax
import jax.numpy as jnp

rows, cols = 5, 5
mapping = jnp.asarray(mapping)       # shape (n, 2)
row_idx, col_idx = mapping.T

# mapping is a constant mapping between tableau indices and grid locations that can be self.mapping in the environment
# tableau will update every iteration so it has to be a parameter

@jax.jit
def stabs_to_grids(tableau):
    # stabs: shape (2 * n,)
    n = mapping.shape[0]

    # Repeat this stabilizer n times: shape (n, 2 * n)
    grid_X = jnp.zeros((rows, cols, n), dtype=tableau.dtype)
    grid_Z = jnp.zeros((rows, cols, n), dtype=tableau.dtype)

    grid_X = grid_X.at[row_idx, col_idx, :].set(tableau[:, :n].T)
    grid_Z = grid_Z.at[row_idx, col_idx, :].set(tableau[:, n:2*n].T)

    # Stack the X and Z along a new axis similar to RGB channels in image, CNNs can then easily associate them
    return jnp.concatenate([grid_X, grid_Z], axis=-1)


grids = stabs_to_grids(tableau)



In [9]:
i = n - 1
print(f"tableau row i (x): {tableau[i][:n]}")
print(f"tableau row i (z): {tableau[i][n:2*n]}")

print(f"grids X (i):")
print(grids[:, :, i])
print(f"grids Z (i):")
print(grids[:, :, i + n])

tableau row i (x): [0 1 1 1 0 1 0 1 1 0 1 0 1 1 0 1 1 1 1 1 1]
tableau row i (z): [0 1 1 1 0 1 0 1 1 0 1 0 1 1 0 1 0 0 0 1 1]
grids X (i):
[[0 0 1 1 0]
 [1 0 1 0 1]
 [1 0 1 0 1]
 [1 0 1 1 1]
 [0 1 1 1 0]]
grids Z (i):
[[0 0 1 1 0]
 [1 0 1 0 1]
 [1 0 1 0 1]
 [1 0 1 0 0]
 [0 0 1 1 0]]


In [10]:
grids.shape
# 42 channels = 21 stabilizers x 2 (X and Z)

(5, 5, 42)

## Test Wrapper
test the wrapper which transforms tableau to grid representation

In [11]:
import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper

# generate 21 stabilizers of length 21
target_21 = [
    "+" + "".join("Z" if row == col else "I" for col in range(21))
    for row in range(21)
]

cliff_gates = CliffordGates(21)
# graph nearest neighbors
graph = []
for ii in range(21):
    if ii > 0:
        graph.append((ii, ii-1))
    if ii < 20:
        graph.append((ii, ii+1))


gates = [cliff_gates.h]
lsp_env = LogicalStatePreparationEnv(target_21, gates=gates, graph=graph)


/Users/shakeeb/dev/rlftqc-biased-noise/.rlftqc-venv/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=uint8 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


TypeError: object of type 'NoneType' has no len()

In [ ]:

lsp_env = TableauToGridWrapper(env=lsp_env, mapping=mapping)
obsv, env_state = lsp_env.reset(key=jax.random.PRNGKey(0))
print(f"observation: {obsv}")
obsv, env_state, reward, terminated, truncated, info = lsp_env.step(key=jax.random.PRNGKey(0), state=env_state, action=5)


In [ ]:
print(f"observation grid: {obsv['grid'].shape}")
print(f"observation phases: {obsv['phases'].shape}")
if "max_diff" in obsv:
    print(f"observation max_diff: {obsv['max_diff'].shape}")

In [ ]:


import jax
import jax.numpy as jnp
import flax.linen as nn
import distrax


NUM_HEADS = 12
NUM_CANDIDATES = 5


class ActorOutput(nn.Module):
    @nn.compact
    def __call__(self, hidden):
        logits = nn.Dense(NUM_HEADS * NUM_CANDIDATES)(hidden)

        # Preserve any leading batch dimensions.
        logits = logits.reshape(
            logits.shape[:-1] + (NUM_HEADS, NUM_CANDIDATES)
        )
        return logits


def test_shapes():
    init_key, sample_key = jax.random.split(jax.random.PRNGKey(0))
    model = ActorOutput()

    # Initialize using dummy actor hidden features.
    params = model.init(init_key, jnp.zeros((4, 128)))

    # Test both an unbatched observation and a batch of four.
    for batch_shape in [(), (4,)]:
        hidden = jnp.zeros(batch_shape + (128,))
        logits = model.apply(params, hidden)

        per_head = distrax.Categorical(logits=logits)
        pi = distrax.Independent(
            per_head,
            reinterpreted_batch_ndims=1,
        )

        sample_key, key = jax.random.split(sample_key)
        actions = pi.sample(seed=key)
        head_log_probs = per_head.log_prob(actions)
        joint_log_prob = pi.log_prob(actions)
        entropy = pi.entropy()

        assert logits.shape == batch_shape + (12, 5)

        assert per_head.batch_shape == batch_shape + (12,)
        assert per_head.event_shape == ()

        assert pi.batch_shape == batch_shape
        assert pi.event_shape == (12,)

        assert actions.shape == batch_shape + (12,)
        assert head_log_probs.shape == batch_shape + (12,)
        assert joint_log_prob.shape == batch_shape
        assert entropy.shape == batch_shape

        assert bool(jnp.all((actions >= 0) & (actions < 5)))
        assert bool(jnp.allclose(
            joint_log_prob,
            head_log_probs.sum(axis=-1),
        ))

        print(f"PASS: batch_shape={batch_shape}")
        print(f"  logits:   {logits.shape}")
        print(f"  actions:  {actions.shape}")
        print(f"  log_prob: {joint_log_prob.shape}")
        print(f"  pi batch: {pi.batch_shape}, event: {pi.event_shape}")
        print(f"per head shape: {per_head.batch_shape}, event: {per_head.event_shape}")


if __name__ == "__main__":
    test_shapes()

## Action Cells
try to organize actions in "cells" so that the independent action spaces aren't mixed up

In [ ]:

# info needed: measure qubit, 4 data qubits, mask 
# create mapping
import numpy as np
n = 21  # number of qubits

rows = 5
cols = 5
corners = [(0, 0), (1, 0), (0, cols - 1),  (0, cols - 2), (rows - 1, 0), (rows - 1, 1), (rows - 1, cols - 1), (rows - 2, cols - 1)]
mapping = []
qubit_index = 0
for i in range(rows):
    for j in range(cols):
        if (i, j) not in corners:
            mapping.append([i, j])
            qubit_index +=1

mapping = np.array(mapping)
print(mapping)
index_grid = -np.ones((rows, cols), dtype=int)
for idx, (i, j) in enumerate(mapping):
    index_grid[i, j] = idx

print(index_grid)


In [ ]:

from dataclasses import dataclass
from typing import NamedTuple


class Neighbors(NamedTuple):
    north: int | None
    east: int | None
    south: int | None
    west: int | None


@dataclass(frozen=True)
class Cell:
    measure_qubit: int
    data_qubits: Neighbors

    # Mask order: center, north, east, south, west
    action_mask: tuple[bool, bool, bool, bool, bool]


In [ ]:
full_cells = [3, 7, 9, 13]
cells = []
for cell_idx in full_cells:
    position = mapping[cell_idx]
    cell = Cell(
        measure_qubit=cell_idx,
        data_qubits=Neighbors(
            north=index_grid[position[0] - 1, position[1]] if position[0] > 0 else None,
            east=index_grid[position[0], position[1] + 1] if position[1] < index_grid.shape[1] - 1 else None,
            south=index_grid[position[0] + 1, position[1]] if position[0] < index_grid.shape[0] - 1 else None,
            west=index_grid[position[0], position[1] - 1] if position[1] > 0 else None
        ),
        action_mask=(True, True, True, True)
    )
    cells.append(cell)

boundary_cells = [0, 5, 16, 11]
for cell_idx in boundary_cells:
    position = mapping[cell_idx]
    cell = Cell(
        measure_qubit=cell_idx,
        data_qubits=Neighbors(
            north=index_grid[position[0] - 1, position[1]] if (position[0] > 0 and index_grid[position[0] - 1, position[1]] != -1) else None,
            east=index_grid[position[0], position[1] + 1] if (position[1] < index_grid.shape[1] - 1 and index_grid[position[0], position[1] + 1] != -1) else None,
            south=index_grid[position[0] + 1, position[1]] if (position[0] < index_grid.shape[0] - 1 and index_grid[position[0] + 1, position[1]] != -1) else None,
            west=index_grid[position[0], position[1] - 1] if (position[1] > 0 and index_grid[position[0], position[1] - 1] != -1) else None
        ),
        # action mask should set nones to false
        action_mask=(
            True if position[0] > 0 and index_grid[position[0] - 1, position[1]] != -1 else False,
            True if position[1] < index_grid.shape[1] - 1 and index_grid[position[0], position[1] + 1] != -1 else False,
            True if position[0] < index_grid.shape[0] - 1 and index_grid[position[0] + 1, position[1]] != -1 else False,
            True if position[1] > 0 and index_grid[position[0], position[1] - 1] != -1 else False
        )
    )
    cells.append(cell)

print(cells)

## Test Env
testing lsp env with grid actions 

In [ ]:
import numpy as np

import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper
from rlftqc.envs.cells import build_cells

# generate 21 stabilizers of length 21
target_21 = [
    "+" + "".join("Z" if row == col else "I" for col in range(21))
    for row in range(21)
]

cliff_gates = CliffordGates(21)
# graph nearest neighbors
graph = []
for ii in range(21):
    if ii > 0:
        graph.append((ii, ii-1))
    if ii < 20:
        graph.append((ii, ii+1))


cells_from_helper = build_cells()

gates = [cliff_gates.h, cliff_gates.cx]
lsp_env = LogicalStatePreparationEnv(target_21, gates=gates, graph=graph, cells=cells_from_helper)


In [ ]:
import jax

key = jax.random.PRNGKey(42)

# random actions
# 20 action vectors, each containing 12 choices from 0 through 4.
fake_actions = jax.random.randint(
    key,
    shape=(20, 12),
    minval=0,
    maxval=5,
)

print(fake_actions)
obs, state = lsp_env.reset_env(key, params=None)

for action in fake_actions: 
    obs, state, reward, terminated, truncated, info = lsp_env.step(key, state, action, params=None)
    print(f"Reward: {reward}, Terminated: {terminated}, Truncated: {truncated}, Info: {info}")

## Full Agent + Env
match action masking of agent with available actions in env and run the loop 

In [18]:
import numpy as np

import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper
from rlftqc.envs.cells import build_cells, get_mapping
from rlftqc.logical_state_preparation import LogicalStatePreparation

# generate 21 stabilizers of length 21
target_21 = [
    "+" + "".join("Z" if row == col else "I" for col in range(17))
    for row in range(17)
]

cliff_gates = CliffordGates(17)

cells_from_helper = build_cells()
mapping = get_mapping()
action_mask = np.zeros((len(cells_from_helper), 5), dtype=int)
for i, cell in enumerate(cells_from_helper):
    action_mask[i, :] = np.array(cell.action_mask)


gates = [cliff_gates.h, cliff_gates.cx]
# lsp_env = LogicalStatePreparationEnv(target_21, gates=gates, graph=graph, cells=cells_from_helper)
lsp = LogicalStatePreparation(target_21, gates=gates, cells=cells_from_helper, action_mask=action_mask, mapping=mapping)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
index_grid:
 [[-1  0  1 -1 -1]
 [-1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 -1]
 [-1 -1 15 16 -1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator

In [19]:
lsp.train()

==== Training begin
obs_tab: (17, 34), obs_sign: (17,)


ValueError: Incompatible shapes for broadcasting: shapes=[(8, 5), (16, 12, 5), ()]

## Create Target

In [15]:
import numpy as np


import sys

sys.path.append(r'../')
from rlftqc.envs.cells import Cell, Neighbors, build_cells, get_mapping

%load_ext autoreload
%autoreload 2
%matplotlib inline

mapping = get_mapping()

target_cells = build_cells()

print(f"mapping: {mapping}")
print(f"target_cells: {target_cells}")

target = []
n = 17
logical_indices = [1, 4, 10]
logical = 'I' * n
paulis = ['X', 'Z']
for i,qubit in enumerate(logical_indices):
    logical = logical[:qubit] + paulis[i % len(paulis)] + logical[qubit+1:]

target.append(logical)

print(f"n: {n}")
measure_qubits = []
for i, cell in enumerate(target_cells):
    measure_qubit = cell.measure_qubit
    data_qubits = cell.data_qubits
    action_mask = np.array(cell.action_mask)
    stab = 'I' * n
    for data_qubit, mask in zip([data_qubits.north, data_qubits.south], action_mask[[1,3]]):
        # print(f"data_qubit: {data_qubit}, mask: {mask}")
        if mask:
            stab = stab[:data_qubit] + 'Z' + stab[data_qubit+1:]

    for data_qubit, mask in zip([data_qubits.east, data_qubits.west], action_mask[[2, 4]]):
        # print(f"data_qubit: {data_qubit}, mask: {mask}")
        if mask:
            stab = stab[:data_qubit] + 'X' + stab[data_qubit+1:]

    # measure qubit must get back positive X by measuring stabilizer
    stab = stab[:measure_qubit] + 'X' + stab[measure_qubit+1:]
    
    target.append(stab)
    print(f"stab for cell {i}: {stab}")
    measure_qubits.append(measure_qubit)

for i, measure_qubit in enumerate(measure_qubits):
    print(f"measure_qubit {i}: {measure_qubit}")
    stabilizer = 'I' * n
    stabilizer = stabilizer[:measure_qubit] + 'X' + stabilizer[measure_qubit+1:]
    target.append(stabilizer)


for stab in target:
    print(f"stab: {stab}")



print(f"len of stabilizers: {len(target)}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
index_grid:
 [[-1  0  1 -1 -1]
 [-1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 -1]
 [-1 -1 15 16 -1]]
mapping: [[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
target_cells: [Cell(measure_qubit=3, data_qubits=Neighbors(north=1, east=4, south=8, west=2), action_mask=(True, True, True, True, True)), Cell(measure_qubit=7, data_qubits=Neighbors(north=2, east=8, south=12, west=6), action_mask=(True, True, True, True, True)), Cell(measure_qubit=9, data_qubits=Neighbors(north=4, east=10, south=14, west=8), action_mask=(True, True, True, True, True)), Cell(measure_qubit=13, data_qubits=Neighbors(north=8, east=14, south=15, west=12), action_mask=(True, True, True, True, True)), Cell(measure_qubit=0, data_qubits=Neighbo

## Full Test
Actual target given to env with agent learning

In [1]:
import numpy as np

import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper
from rlftqc.envs.cells import build_cells, get_mapping, get_target
from rlftqc.logical_state_preparation import LogicalStatePreparation

target = get_target()

print(f"len(target): {len(target)}")

cliff_gates = CliffordGates(17)

cells_from_helper = build_cells()
mapping = get_mapping()
action_mask = np.zeros((len(cells_from_helper), 5), dtype=int)
for i, cell in enumerate(cells_from_helper):
    action_mask[i, :] = np.array(cell.action_mask)


gates = [cliff_gates.h, cliff_gates.cx]
lsp = LogicalStatePreparation(target, gates=gates, cells=cells_from_helper, action_mask=action_mask, mapping=mapping)


[[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
index_grid:
 [[-1  0  1 -1 -1]
 [-1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 -1]
 [-1 -1 15 16 -1]]
mapping: [[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
target_cells: [Cell(measure_qubit=3, data_qubits=Neighbors(north=1, east=4, south=8, west=2), action_mask=(True, True, True, True, True)), Cell(measure_qubit=7, data_qubits=Neighbors(north=2, east=8, south=12, west=6), action_mask=(True, True, True, True, True)), Cell(measure_qubit=9, data_qubits=Neighbors(north=4, east=10, south=14, west=8), action_mask=(True, True, True, True, True)), Cell(measure_qubit=13, data_qubits=Neighbors(north=8, east=14, south=15, west=12), action_mask=(True, True, True, True, True)), Cell(measure_qubit=0, data_qubits=Neighbors(north=None, east=1, south=2, west=None), action_mask=(True, False, True, True, False)

/Users/shakeeb/dev/rlftqc-biased-noise/.rlftqc-venv/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=uint8 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]
17
h operator: [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [

In [2]:
lsp.train()

==== Training begin
obs_tab: (17, 34), obs_sign: (17,)
obs_tab: (17, 34), obs_sign: (17,)
obs_tab: (17, 34), obs_sign: (17,)
==== Training finished, time elapsed: 2849.26333 s


In [3]:
lsp.run()

==== No circuits found.
Tips: 
(1) Try to change the max_steps parameter to increase the number of possible gates in the circuit.
(2) Change the training configuration by increasing the TOTAL_TIMESTEPS to train longer. For example: training_config['TOTAL_TIMESTEPS'] = 1e6.
(3) Change the other training configurations.


In [4]:
lsp.log()

==== Logging results in folder:  results-lsp/17/


TypeError: Object of type ndarray is not JSON serializable

## Correct Target
Simulation of environment under the correct syndrome extraction circuits in order to find out if target state is correct

In [ ]:
import numpy as np

import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper
from rlftqc.envs.cells import build_cells, get_mapping, get_target
from rlftqc.logical_state_preparation import LogicalStatePreparation

target = get_target()

print(f"len(target): {len(target)}")

cliff_gates = CliffordGates(17)

cells_from_helper = build_cells()
mapping = get_mapping()
action_mask = np.zeros((len(cells_from_helper), 5), dtype=int)
for i, cell in enumerate(cells_from_helper):
    action_mask[i, :] = np.array(cell.action_mask)


gates = [cliff_gates.h, cliff_gates.cx]
lsp = LogicalStatePreparation(target, gates=gates, cells=cells_from_helper, action_mask=action_mask, mapping=mapping)


